# Cohezion: ARC-AGI Epistemic Humility Evaluator
This notebook evaluates models against ARC-AGI style grid patterns using FLUME latent state tracking.

## Methodology
We test if models can identify 'Insufficient Information' in ambiguous grid transformations.

In [ ]:
# Offline setup: Creating local cohezion package structure
import os
from pathlib import Path


os.makedirs('cohezion/flume', exist_ok=True)
Path('cohezion/__init__.py').touch()

COHEZION_BUNDLE = {"flume/__init__.py": "\"\"\"FLUME VAE module for semantic embeddings and latent space operations.\"\"\"\n\nfrom cohezion.flume.vae_encoder import FlumeVAEEncoder\n\n\n__all__ = [\n    \"ExperienceDataset\",\n    \"ExperienceEncoder\",\n    \"ExperienceTrainingPipeline\",\n    \"FlumeVAEEncoder\",\n]\n\n\ndef __getattr__(name: str):\n    \"\"\"Lazy imports for experience pipeline classes.\"\"\"\n    if name == \"ExperienceEncoder\":\n        from cohezion.flume.experience_encoder import ExperienceEncoder\n\n        return ExperienceEncoder\n    if name == \"ExperienceDataset\":\n        from cohezion.flume.experience_dataset import ExperienceDataset\n\n        return ExperienceDataset\n    if name == \"ExperienceTrainingPipeline\":\n        from cohezion.flume.experience_pipeline import ExperienceTrainingPipeline\n\n        return ExperienceTrainingPipeline\n    raise AttributeError(f\"module {__name__!r} has no attribute {name!r}\")\n", "flume/grid_encoder.py": "\"\"\"FLUME Grid Encoder for ARC-AGI style grid patterns.\n\nSpecialized encoder/decoder for 2D matrices (0-9) representing color grids.\nMaps grids to 256D FLUME latent space for semantic reasoning and trajectory tracking.\n\"\"\"\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom typing import List, Tuple, Optional\n\n\nclass ARCGridEncoder(nn.Module):\n    \"\"\"\n    Encoder for ARC-AGI grids.\n    Uses a simple CNN or MLP to project 2D grids into 256D.\n    \"\"\"\n    def __init__(self, latent_dim: int = 256, max_grid_size: int = 30):\n        super().__init__()\n        self.latent_dim = latent_dim\n        self.max_grid_size = max_grid_size\n        \n        # Flattened input size (max_grid_size * max_grid_size)\n        # We use padding for smaller grids\n        self.input_dim = max_grid_size * max_grid_size\n        \n        self.encoder = nn.Sequential(\n            nn.Linear(self.input_dim, 512),\n            nn.ReLU(),\n            nn.Linear(512, 512),\n            nn.ReLU(),\n            nn.Linear(512, latent_dim)\n        )\n        \n        self.decoder = nn.Sequential(\n            nn.Linear(latent_dim, 512),\n            nn.ReLU(),\n            nn.Linear(512, 512),\n            nn.ReLU(),\n            nn.Linear(512, self.input_dim),\n            nn.Sigmoid() # Output normalized probabilities/intensities for 0-9\n        )\n\n    def preprocess_grid(self, grid: List[List[int]]) -> torch.Tensor:\n        \"\"\"Pad and flatten grid for encoding.\"\"\"\n        rows = len(grid)\n        cols = len(grid[0]) if rows > 0 else 0\n        \n        flat_grid = np.zeros((self.max_grid_size, self.max_grid_size), dtype=np.float32)\n        \n        # Fill existing grid into top-left\n        for r in range(min(rows, self.max_grid_size)):\n            for c in range(min(cols, self.max_grid_size)):\n                # Normalize color 0-9 to 0.0-0.9\n                flat_grid[r, c] = grid[r][c] / 10.0\n                \n        return torch.from_numpy(flat_grid.flatten()).unsqueeze(0)\n\n    def encode(self, grid: List[List[int]]) -> torch.Tensor:\n        \"\"\"Encode 2D grid to latent vector.\"\"\"\n        x = self.preprocess_grid(grid)\n        return self.encoder(x)\n\n    def decode(self, z: torch.Tensor, original_shape: Tuple[int, int]) -> List[List[int]]:\n        \"\"\"Decode latent vector back to 2D grid of specific shape.\"\"\"\n        x_hat = self.decoder(z)\n        x_hat = x_hat.view(self.max_grid_size, self.max_grid_size).detach().cpu().numpy()\n        \n        rows, cols = original_shape\n        grid = []\n        for r in range(rows):\n            row = []\n            for c in range(cols):\n                # Denormalize and round to nearest integer 0-9\n                val = int(round(x_hat[r, c] * 10.0))\n                row.append(max(0, min(9, val)))\n            grid.append(row)\n        return grid\n\n\nclass FlumeGridHarness:\n    \"\"\"\n    Harness for integrating Grid Encoding into the FLUME evaluation loop.\n    \"\"\"\n    def __init__(self, device: str = \"cpu\"):\n        self.device = device\n        self.model = ARCGridEncoder().to(device)\n        self.model.eval()\n\n    def get_grid_embedding(self, grid_str: str) -> np.ndarray:\n        \"\"\"\n        Parses a grid string (e.g. \"[[1,2],[3,4]]\") and returns its 256D embedding.\n        \"\"\"\n        try:\n            grid = json.loads(grid_str)\n            with torch.no_grad():\n                z = self.model.encode(grid)\n                return z.squeeze(0).cpu().numpy()\n        except Exception:\n            # Fallback to zero vector or hash if parsing fails\n            return np.zeros(256, dtype=np.float32)\n\nimport json\n"}

for path, content in COHEZION_BUNDLE.items():
    with open(f'cohezion/{path}', 'w') as f:
        f.write(content)

print('Cohezion bundle initialized.')

In [ ]:
# Install dependencies (if internet is enabled, or use attached wheels)
try:
    import numpy
    import torch
    print('Found core ML libraries.')
except ImportError:
    !pip install -q torch numpy transformers accelerate

In [ ]:
import json

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from cohezion.flume.grid_encoder import FlumeGridHarness


# Assuming benchmark dataset is attached to the kernel
BENCHMARK_FILE = '../input/cohezion-agi-benchmark/evo_hiho_benchmark.json'
if not os.path.exists(BENCHMARK_FILE):
    BENCHMARK_FILE = 'evo_hiho_benchmark.json' # Local fallback

def load_benchmark():
    with open(BENCHMARK_FILE, 'r') as f:
        return json.load(f)

benchmark_data = load_benchmark()
tasks = benchmark_data.get('train', []) + benchmark_data.get('test', [])
print(f"Loaded {len(tasks)} tasks.")

In [ ]:
def evaluate_model(model_id, model_name):
    print(f"\n--- Evaluating {model_name} ---")
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        device_map='auto', 
        torch_dtype=torch.float16,
        trust_remote_code=True
    )
    
    harness = FlumeGridHarness()
    
    correct = 0
    total = len(tasks)
    results = []
    
    for task in tasks:
        prompt = f"Answer the ARC grid problem. Output only the selected option or state if insufficient.\n\n{task['input']}"
        target = task['output']
        
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=50)
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        passed = target.lower() in response.lower() or 'insufficient information' in response.lower()
        if passed:
            correct += 1
            
        results.append({
            'passed': passed,
            'response': response
        })
            
    print(f"Accuracy for {model_name}: {correct}/{total} ({(correct/total)*100:.2f}%)")
    return correct / total

In [ ]:
# Example evaluation (Qwen 2.5 is strong on ARC)
# qwen_score = evaluate_model('Qwen/Qwen2.5-7B-Instruct', 'Qwen2.5-7B-Instruct')